In [ ]:
#!/usr/bin/env python3
"""
Comparative Genomic Distribution Analysis of DMGs
===================================================
Determines whether CpG DMGs and CpH DMGs are located in similar or distinct
genomic contexts, indicating shared vs distinct regulatory mechanisms.

Includes length-matched control analysis to determine if feature enrichment
is independent of gene length.

Genome: rheMac10 (Macaca mulatta)
"""

import pandas as pd
import numpy as np
import pyranges as pr
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu
from collections import OrderedDict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================

MCG_BED = "all_dmgs_mcg.bed"
MCH_BED = "all_dmgs_mch.bed"
GTF_FILE = "../globalmethyl/Macaca_mulatta.Mmul_10.104.gtf"

PROMOTER_UPSTREAM = 2000
PROMOTER_DOWNSTREAM = 500

# Number of random length-matched background samples (for robust p-values)
N_PERMUTATIONS = 1000

OUTPUT_DIR = "genomic_context_results"

FONT_FAMILY = "Arial"
FONT_SIZE_TITLE = 14
FONT_SIZE_LABEL = 12
FONT_SIZE_TICK = 10
DPI = 300

FEATURE_ORDER = ["Promoter", "5'UTR", "Exon", "Intron", "3'UTR"]
PALETTE = OrderedDict([
    ("Promoter",   "#e74c3c"),
    ("5'UTR",      "#e67e22"),
    ("Exon",       "#2ecc71"),
    ("Intron",     "#3498db"),
    ("3'UTR",      "#f1c40f"),
])

# =============================================================================
# CHROMOSOME HARMONIZATION
# =============================================================================

def has_chr_prefix(pr_obj):
    return pr_obj.df["Chromosome"].astype(str).str.startswith("chr").any()

def strip_chr(df):
    df = df.copy()
    df["Chromosome"] = df["Chromosome"].astype(str).str.replace("^chr", "", regex=True)
    return df

def add_chr(df):
    df = df.copy()
    mask = ~df["Chromosome"].astype(str).str.startswith("chr")
    df.loc[mask, "Chromosome"] = "chr" + df.loc[mask, "Chromosome"].astype(str)
    return df

# =============================================================================
# BUILD GENOMIC FEATURES
# =============================================================================

def build_feature_regions(gtf_path, promoter_up, promoter_down):
    print(f"Parsing GTF: {gtf_path}")
    gtf = pr.read_gtf(gtf_path)
    gtf_chr = has_chr_prefix(gtf)
    print(f"  GTF chr prefix: {gtf_chr}")

    genes = gtf[gtf.Feature == "gene"]
    exons = gtf[gtf.Feature == "exon"]
    transcripts = gtf[gtf.Feature == "transcript"]
    utr5 = gtf[gtf.Feature == "five_prime_utr"]
    utr3 = gtf[gtf.Feature == "three_prime_utr"]

    print(f"  Genes: {len(genes)}, Exons: {len(exons)}, Transcripts: {len(transcripts)}")
    print(f"  5'UTRs: {len(utr5)}, 3'UTRs: {len(utr3)}")

    tx_df = transcripts.df.copy()
    ps = np.where(tx_df["Strand"] == "+",
                  tx_df["Start"] - promoter_up,
                  tx_df["End"] - promoter_down)
    pe = np.where(tx_df["Strand"] == "+",
                  tx_df["Start"] + promoter_down,
                  tx_df["End"] + promoter_up)
    promoters = pr.PyRanges(pd.DataFrame({
        "Chromosome": tx_df["Chromosome"].values,
        "Start": np.maximum(0, ps), "End": pe
    })).merge()

    exons_m = exons.merge()
    introns = genes.subtract(exons_m)
    utr5_m = utr5.merge() if len(utr5) > 0 else pr.PyRanges()
    utr3_m = utr3.merge() if len(utr3) > 0 else pr.PyRanges()

    print("  Building priority-based features...")
    features = OrderedDict()
    features["Promoter"] = promoters

    features["5'UTR"] = utr5_m.subtract(promoters) if len(utr5_m) > 0 else pr.PyRanges()

    if len(utr3_m) > 0:
        u3 = utr3_m.subtract(promoters)
        if len(utr5_m) > 0:
            u3 = u3.subtract(utr5_m)
        features["3'UTR"] = u3
    else:
        features["3'UTR"] = pr.PyRanges()

    ex = exons_m.subtract(promoters)
    if len(utr5_m) > 0: ex = ex.subtract(utr5_m)
    if len(utr3_m) > 0: ex = ex.subtract(utr3_m)
    features["Exon"] = ex

    intr = introns.subtract(promoters)
    if len(utr5_m) > 0: intr = intr.subtract(utr5_m)
    if len(utr3_m) > 0: intr = intr.subtract(utr3_m)
    intr = intr.subtract(exons_m)
    features["Intron"] = intr

    for name, feat in features.items():
        bp = int((feat.df["End"] - feat.df["Start"]).sum()) if len(feat) > 0 else 0
        print(f"    {name}: {len(feat):,} regions, {bp:,} bp")

    return features, genes, gtf_chr

# =============================================================================
# BASE-PAIR PROPORTION CALCULATION
# =============================================================================

def compute_bp_overlap(gene_start, gene_end, feat_starts, feat_ends):
    ov_start = np.maximum(feat_starts, gene_start)
    ov_end = np.minimum(feat_ends, gene_end)
    return int(np.maximum(0, ov_end - ov_start).sum())


def compute_gene_feature_proportions(genes_df, features):
    feat_by_chrom = {}
    for fname, fpr in features.items():
        if len(fpr) == 0:
            feat_by_chrom[fname] = {}
            continue
        fdf = fpr.df[["Chromosome", "Start", "End"]]
        feat_by_chrom[fname] = {}
        for c, grp in fdf.groupby("Chromosome"):
            feat_by_chrom[fname][c] = (grp["Start"].values, grp["End"].values)

    results = []
    total = len(genes_df)

    for i, (_, row) in enumerate(genes_df.iterrows()):
        if (i + 1) % 5000 == 0:
            print(f"    Processing gene {i+1}/{total}...")

        chrom, start, end = row["Chromosome"], row["Start"], row["End"]
        gene_len = end - start
        if gene_len <= 0:
            continue

        gene_result = {"Chromosome": chrom, "Start": start, "End": end,
                       "gene_length": gene_len}

        total_assigned = 0
        for fname in FEATURE_ORDER:
            chrom_data = feat_by_chrom.get(fname, {}).get(chrom, None)
            if chrom_data is None:
                bp = 0
            else:
                bp = compute_bp_overlap(start, end, chrom_data[0], chrom_data[1])
            gene_result[f"{fname}_bp"] = bp
            gene_result[f"{fname}_pct"] = bp / gene_len * 100
            total_assigned += bp

        gene_result["Other_bp"] = max(0, gene_len - total_assigned)
        gene_result["Other_pct"] = max(0, gene_len - total_assigned) / gene_len * 100
        results.append(gene_result)

    return pd.DataFrame(results)

# =============================================================================
# LENGTH-MATCHED BACKGROUND
# =============================================================================

def get_length_matched_background(dmg_props, all_props, n_samples=1000, seed=42):
    """
    For each DMG, find all genes in the background with similar length
    (within 2-fold), then sample a matched background set.
    Repeat n_samples times and return aggregated proportions for each sample.
    """
    rng = np.random.RandomState(seed)
    dmg_lengths = dmg_props["gene_length"].values
    all_lengths = all_props["gene_length"].values
    all_indices = np.arange(len(all_props))

    # Pre-compute log lengths for efficient matching
    log_dmg = np.log2(dmg_lengths)
    log_all = np.log2(all_lengths)

    sampled_aggs = []

    for perm in range(n_samples):
        matched_indices = []
        for dl in log_dmg:
            # Find genes within 2-fold (1 log2 unit)
            candidates = all_indices[np.abs(log_all - dl) <= 1.0]
            if len(candidates) == 0:
                # Relax to 4-fold
                candidates = all_indices[np.abs(log_all - dl) <= 2.0]
            if len(candidates) == 0:
                continue
            matched_indices.append(rng.choice(candidates))

        if len(matched_indices) == 0:
            continue

        matched_df = all_props.iloc[matched_indices]
        total_bp = matched_df["gene_length"].sum()

        agg = {}
        for fname in FEATURE_ORDER:
            feat_bp = matched_df[f"{fname}_bp"].sum()
            agg[fname] = feat_bp / total_bp * 100 if total_bp > 0 else 0
        sampled_aggs.append(agg)

    return sampled_aggs


def length_matched_enrichment_test(dmg_agg, matched_aggs, dmg_label="DMG"):
    """
    Compare DMG feature proportions against the distribution of
    length-matched background samples.
    Empirical p-value = fraction of permutations where background >= DMG.
    """
    results = []
    for fname in FEATURE_ORDER:
        dmg_pct = dmg_agg[fname]["pct_of_total_bp"]
        bg_pcts = [agg[fname] for agg in matched_aggs]
        bg_mean = np.mean(bg_pcts)
        bg_std = np.std(bg_pcts)

        # Two-sided empirical p-value
        n_extreme = sum(1 for bp in bg_pcts if abs(bp - bg_mean) >= abs(dmg_pct - bg_mean))
        p_empirical = n_extreme / len(bg_pcts)

        fold = dmg_pct / bg_mean if bg_mean > 0 else (np.inf if dmg_pct > 0 else 0)

        results.append({
            "Feature": fname,
            f"{dmg_label}_pct": round(dmg_pct, 2),
            "LengthMatched_mean_pct": round(bg_mean, 2),
            "LengthMatched_std_pct": round(bg_std, 2),
            "Fold_vs_matched": round(fold, 3),
            "Empirical_p": round(p_empirical, 4),
            "Significant": p_empirical < 0.05
        })

    return pd.DataFrame(results)

# =============================================================================
# AGGREGATE & COMPARE
# =============================================================================

def aggregate_proportions(prop_df, label=""):
    result = OrderedDict()
    total_bp = prop_df["gene_length"].sum()

    for fname in FEATURE_ORDER:
        feat_bp = prop_df[f"{fname}_bp"].sum()
        result[fname] = {
            "total_bp": int(feat_bp),
            "pct_of_total_bp": feat_bp / total_bp * 100 if total_bp > 0 else 0,
            "mean_pct_per_gene": prop_df[f"{fname}_pct"].mean(),
            "median_pct_per_gene": prop_df[f"{fname}_pct"].median(),
            "std_pct_per_gene": prop_df[f"{fname}_pct"].std(),
        }

    if label:
        print(f"\n  {label} (n={len(prop_df)} genes, {total_bp:,} total bp):")
        for fname, vals in result.items():
            print(f"    {fname:>10s}: {vals['pct_of_total_bp']:6.2f}% of bp, "
                  f"mean {vals['mean_pct_per_gene']:5.1f}% per gene")
    return result


def enrichment_test_features(dmg_props, all_props, dmg_label="DMG"):
    results = []
    dmg_total = sum(v["total_bp"] for v in dmg_props.values())
    all_total = sum(v["total_bp"] for v in all_props.values())

    for fname in FEATURE_ORDER:
        a = dmg_props[fname]["total_bp"]
        b = dmg_total - a
        c = all_props[fname]["total_bp"]
        d = all_total - c

        dmg_pct = dmg_props[fname]["pct_of_total_bp"]
        all_pct = all_props[fname]["pct_of_total_bp"]
        fold = dmg_pct / all_pct if all_pct > 0 else (np.inf if dmg_pct > 0 else 0)

        try:
            odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative='two-sided')
        except:
            odds_ratio, pval = np.nan, 1.0

        results.append({
            "Feature": fname,
            f"{dmg_label}_bp": a,
            f"{dmg_label}_pct": round(dmg_pct, 2),
            "AllGenes_bp": c,
            "AllGenes_pct": round(all_pct, 2),
            "Fold_enrichment": round(fold, 3),
            "Odds_ratio": round(odds_ratio, 3) if not np.isnan(odds_ratio) else np.nan,
            "P_value": pval
        })

    df = pd.DataFrame(results)
    n = len(df[df[f"{dmg_label}_bp"] + df["AllGenes_bp"] > 0])
    df["P_adjusted"] = np.minimum(df["P_value"] * n, 1.0)
    df["Significant"] = df["P_adjusted"] < 0.05
    return df


def compare_mcg_vs_mch(mcg_props, mch_props):
    mcg_bp = [mcg_props[f]["total_bp"] for f in FEATURE_ORDER]
    mch_bp = [mch_props[f]["total_bp"] for f in FEATURE_ORDER]
    contingency = np.array([mcg_bp, mch_bp])
    nonzero = contingency.sum(axis=0) > 0
    contingency_clean = contingency[:, nonzero]

    if contingency_clean.shape[1] < 2:
        print("  Not enough features for chi-square test.")
        return None, None

    chi2, chi2_p, dof, expected = chi2_contingency(contingency_clean)
    print(f"\n  Chi-square test (mCG vs mCH feature distribution):")
    print(f"    chi2={chi2:.2f}, df={dof}, p={chi2_p:.2e}")

    results = []
    mcg_total = sum(mcg_bp)
    mch_total = sum(mch_bp)

    for fname in FEATURE_ORDER:
        a = mcg_props[fname]["total_bp"]
        b = mcg_total - a
        c = mch_props[fname]["total_bp"]
        d = mch_total - c

        try:
            odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative='two-sided')
        except:
            odds_ratio, pval = np.nan, 1.0

        results.append({
            "Feature": fname,
            "mCG_pct": round(mcg_props[fname]["pct_of_total_bp"], 2),
            "mCH_pct": round(mch_props[fname]["pct_of_total_bp"], 2),
            "Odds_ratio": round(odds_ratio, 3) if not np.isnan(odds_ratio) else np.nan,
            "P_value": pval
        })

    df = pd.DataFrame(results)
    df["P_adjusted"] = np.minimum(df["P_value"] * len(df), 1.0)
    df["Significant"] = df["P_adjusted"] < 0.05
    return (chi2, chi2_p, dof), df

# =============================================================================
# PLOTTING
# =============================================================================

def setup_rc():
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': [FONT_FAMILY, 'Helvetica', 'DejaVu Sans'],
        'font.size': FONT_SIZE_TICK,
        'axes.linewidth': 1.0,
    })


def plot_stacked_comparison(mcg_agg, mch_agg, all_agg, output_path):
    setup_rc()

    groups = ["mCG DMGs", "mCH DMGs", "All Genes\n(rheMac10)"]
    aggs = [mcg_agg, mch_agg, all_agg]

    fig, ax = plt.subplots(figsize=(7, 5.5))

    x = np.arange(len(groups))
    width = 0.55
    bottoms = np.zeros(len(groups))

    for fname in FEATURE_ORDER:
        vals = [agg[fname]["pct_of_total_bp"] for agg in aggs]
        color = PALETTE[fname]
        ax.bar(x, vals, width, bottom=bottoms, color=color,
               edgecolor='white', linewidth=0.5)

        for i, v in enumerate(vals):
            if v > 5:
                ax.text(x[i], bottoms[i] + v / 2, f'{v:.1f}%',
                        ha='center', va='center', fontsize=FONT_SIZE_TICK,
                        fontweight='bold', color='white')
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels(groups, fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_ylabel("Percentage of total gene body base pairs (%)",
                  fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_title("Genomic Feature Distribution",
                 fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
    ax.set_ylim(0, 105)

    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=PALETTE[f], edgecolor='white')
               for f in FEATURE_ORDER]
    ax.legend(handles, FEATURE_ORDER,
              loc='upper left', bbox_to_anchor=(1.02, 1.0),
              fontsize=FONT_SIZE_TICK, frameon=False, title="Percent of gene body by feature",
              title_fontsize=FONT_SIZE_TICK + 1)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.savefig(output_path.replace('.pdf', '.png'), dpi=DPI, bbox_inches='tight')
    print(f"  Saved: {output_path}")
    plt.close()


def plot_fold_enrichment_comparison(mcg_enrich, mch_enrich, output_path):
    setup_rc()

    mcg_data = mcg_enrich.set_index("Feature")
    mch_data = mch_enrich.set_index("Feature")

    features = [f for f in FEATURE_ORDER
                if f in mcg_data.index and f in mch_data.index]

    mcg_fold = np.array([mcg_data.loc[f, "Fold_enrichment"] for f in features])
    mch_fold = np.array([mch_data.loc[f, "Fold_enrichment"] for f in features])
    mcg_padj = np.array([mcg_data.loc[f, "P_adjusted"] for f in features])
    mch_padj = np.array([mch_data.loc[f, "P_adjusted"] for f in features])

    mask = (mcg_fold > 0) | (mch_fold > 0)
    features = [f for f, m in zip(features, mask) if m]
    mcg_fold = mcg_fold[mask]
    mch_fold = mch_fold[mask]
    mcg_padj = mcg_padj[mask]
    mch_padj = mch_padj[mask]

    if len(features) == 0:
        print("  WARNING: No features to plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    x = np.arange(len(features))
    width = 0.35

    ax.bar(x - width/2, mcg_fold, width, label='mCG DMGs',
           color='#e74c3c', edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.bar(x + width/2, mch_fold, width, label='mCH DMGs',
           color='#3498db', edgecolor='black', linewidth=0.5, alpha=0.85)

    ax.axhline(y=1, color="black", linewidth=0.8, linestyle="--", alpha=0.7)

    max_fold = max(max(mcg_fold), max(mch_fold), 1.5)
    for i in range(len(features)):
        star = "***" if mcg_padj[i] < 0.001 else "**" if mcg_padj[i] < 0.01 else "*" if mcg_padj[i] < 0.05 else ""
        if star:
            ax.text(x[i] - width/2, mcg_fold[i] + 0.03 * max_fold, star,
                    ha='center', va='bottom', fontsize=8, fontweight='bold',
                    color='#e74c3c')

        star = "***" if mch_padj[i] < 0.001 else "**" if mch_padj[i] < 0.01 else "*" if mch_padj[i] < 0.05 else ""
        if star:
            ax.text(x[i] + width/2, mch_fold[i] + 0.03 * max_fold, star,
                    ha='center', va='bottom', fontsize=8, fontweight='bold',
                    color='#3498db')

    ax.set_xticks(x)
    ax.set_xticklabels(features, fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_ylabel("Fold Enrichment vs All Genes",
                  fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_title("Genomic Feature Enrichment:\nmCG DMGs vs mCH DMGs",
                 fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
    ax.legend(fontsize=FONT_SIZE_TICK, frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.savefig(output_path.replace('.pdf', '.png'), dpi=DPI, bbox_inches='tight')
    print(f"  Saved: {output_path}")
    plt.close()


def plot_per_gene_boxplots(mcg_prop_df, mch_prop_df, all_prop_df, output_path):
    setup_rc()

    fig, axes = plt.subplots(1, len(FEATURE_ORDER),
                              figsize=(3 * len(FEATURE_ORDER), 4),
                              sharey=False)

    for ax, fname in zip(axes, FEATURE_ORDER):
        col = f"{fname}_pct"
        data = [mcg_prop_df[col].values, mch_prop_df[col].values,
                all_prop_df[col].values]

        bp = ax.boxplot(data, labels=["mCG\nDMGs", "mCH\nDMGs", "All\nGenes"],
                        patch_artist=True, widths=0.6,
                        medianprops=dict(color='black', linewidth=1.5),
                        flierprops=dict(markersize=3, alpha=0.5))

        colors = ['#e74c3c', '#3498db', '#95a5a6']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_title(fname, fontsize=FONT_SIZE_LABEL, fontweight='bold')
        ax.set_ylabel("% of gene" if ax == axes[0] else "")
        ax.tick_params(axis='x', labelsize=FONT_SIZE_TICK - 1)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle("Per-Gene Feature Proportions",
                 fontsize=FONT_SIZE_TITLE, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.savefig(output_path.replace('.pdf', '.png'), dpi=DPI, bbox_inches='tight')
    print(f"  Saved: {output_path}")
    plt.close()


def plot_gene_length_comparison(mcg_prop_df, mch_prop_df, all_prop_df, output_path):
    """Gene length boxplot with fixed label placement."""
    setup_rc()

    fig, ax = plt.subplots(figsize=(5, 4.5))

    data = [
        mcg_prop_df["gene_length"].values / 1000,
        mch_prop_df["gene_length"].values / 1000,
        all_prop_df["gene_length"].values / 1000,
    ]

    labels = [
        f"mCG DMGs\n(n={len(mcg_prop_df)})",
        f"mCH DMGs\n(n={len(mch_prop_df)})",
        f"All Genes\n(n={len(all_prop_df):,})"
    ]

    bp = ax.boxplot(data, labels=labels,
                    patch_artist=True, widths=0.6,
                    medianprops=dict(color='black', linewidth=1.5),
                    flierprops=dict(markersize=2, alpha=0.3),
                    showfliers=True)

    colors = ['#e74c3c', '#3498db', '#95a5a6']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # Place median labels ABOVE the box, offset to the right
    for i, d in enumerate(data):
        median_val = np.median(d)
        # Get the top of the box (Q3)
        q3 = np.percentile(d, 75)
        ax.annotate(f'{median_val:.1f} kb',
                    xy=(i + 1, median_val),
                    xytext=(i + 1.35, median_val),
                    fontsize=FONT_SIZE_TICK, fontweight='bold',
                    ha='center', va='center',
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8))
    
    ax.set_xticklabels(labels, fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_ylabel("Gene Length (kb)", fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_title("Gene Length Distribution",
                 fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
    ax.set_yscale('log')
    ax.tick_params(axis='x', labelsize=FONT_SIZE_TICK)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Add Mann-Whitney U test results
    #_, p_mcg_all = mannwhitneyu(data[0], data[2], alternative='two-sided')
   # _, p_mch_all = mannwhitneyu(data[1], data[2], alternative='two-sided')
   # _, p_mcg_mch = mannwhitneyu(data[0], data[1], alternative='two-sided')

   # stats_text = (f"mCG vs All: p={p_mcg_all:.2e}\n"
    #              f"mCH vs All: p={p_mch_all:.2e}\n"
    #              f"mCG vs mCH: p={p_mcg_mch:.2e}")
   # ax.text(0.02, 0.02, stats_text, transform=ax.transAxes,
    #        fontsize=FONT_SIZE_TICK - 2, verticalalignment='bottom',
    #        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
     #                 edgecolor='gray', alpha=0.8))

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.savefig(output_path.replace('.pdf', '.png'), dpi=DPI, bbox_inches='tight')
    print(f"  Saved: {output_path}")
    plt.close()


def plot_length_matched_comparison(mcg_agg, mch_agg, mcg_matched_mean, mch_matched_mean,
                                   output_path):
    """
    Stacked bars: DMGs vs their length-matched backgrounds.
    Shows whether enrichment persists after controlling for gene length.
    """
    setup_rc()

    groups = ["mCG\nDMGs", "mCG\nLength-matched", "mCH\nDMGs", "mCH\nLength-matched"]

    fig, ax = plt.subplots(figsize=(8, 5.5))

    x = np.arange(len(groups))
    width = 0.55
    bottoms = np.zeros(len(groups))

    for fname in FEATURE_ORDER:
        vals = [
            mcg_agg[fname]["pct_of_total_bp"],
            mcg_matched_mean[fname],
            mch_agg[fname]["pct_of_total_bp"],
            mch_matched_mean[fname],
        ]
        color = PALETTE[fname]
        ax.bar(x, vals, width, bottom=bottoms, color=color,
               edgecolor='white', linewidth=0.5)

        for i, v in enumerate(vals):
            if v > 5:
                ax.text(x[i], bottoms[i] + v / 2, f'{v:.1f}%',
                        ha='center', va='center', fontsize=FONT_SIZE_TICK - 1,
                        fontweight='bold', color='white')
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels(groups, fontsize=FONT_SIZE_LABEL - 1, fontweight='bold')
    ax.set_ylabel("Percentage of total gene body base pairs (%)",
                  fontsize=FONT_SIZE_LABEL, fontweight='bold')
    ax.set_title("DMGs vs Length-Matched Background\n(controlling for gene length)",
                 fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
    ax.set_ylim(0, 105)

    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=PALETTE[f], edgecolor='white')
               for f in FEATURE_ORDER]
    ax.legend(handles, FEATURE_ORDER,
              loc='upper left', bbox_to_anchor=(1.02, 1.0),
              fontsize=FONT_SIZE_TICK, frameon=False, title="Percent of gene body by feature",
              title_fontsize=FONT_SIZE_TICK + 1)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.savefig(output_path.replace('.pdf', '.png'), dpi=DPI, bbox_inches='tight')
    print(f"  Saved: {output_path}")
    plt.close()

# =============================================================================
# MAIN
# =============================================================================

def main():
    for f, label in [(MCG_BED, "mCG BED"), (MCH_BED, "mCH BED"), (GTF_FILE, "GTF")]:
        if not os.path.exists(f):
            sys.exit(f"ERROR: {label} not found: {f}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load DMGs
    print("Loading DMG files...")
    mcg_df = pd.read_csv(MCG_BED, sep="\t", header=None, comment="#",
                         usecols=[0, 1, 2], names=["Chromosome", "Start", "End"])
    mch_df = pd.read_csv(MCH_BED, sep="\t", header=None, comment="#",
                         usecols=[0, 1, 2], names=["Chromosome", "Start", "End"])

    print(f"  mCG DMGs: {len(mcg_df)}")
    print(f"  mCH DMGs: {len(mch_df)}")

    # Build features
    features, genes, gtf_has_chr = build_feature_regions(
        GTF_FILE, PROMOTER_UPSTREAM, PROMOTER_DOWNSTREAM)

    # Harmonize chromosomes
    print("\nHarmonizing chromosome names...")
    dmr_has_chr = mcg_df["Chromosome"].astype(str).str.startswith("chr").any()
    if dmr_has_chr and not gtf_has_chr:
        print("  -> Stripping 'chr' prefix from DMG coordinates")
        mcg_df = strip_chr(mcg_df)
        mch_df = strip_chr(mch_df)
    elif not dmr_has_chr and gtf_has_chr:
        print("  -> Adding 'chr' prefix to DMG coordinates")
        mcg_df = add_chr(mcg_df)
        mch_df = add_chr(mch_df)

    all_genes_df = genes.df[["Chromosome", "Start", "End"]].drop_duplicates().copy()
    print(f"  All genes in GTF: {len(all_genes_df)}")

    # Compute proportions
    print("\nComputing feature proportions for mCG DMGs...")
    mcg_props = compute_gene_feature_proportions(mcg_df, features)

    print("\nComputing feature proportions for mCH DMGs...")
    mch_props = compute_gene_feature_proportions(mch_df, features)

    print("\nComputing feature proportions for all genes...")
    all_props = compute_gene_feature_proportions(all_genes_df, features)

    # Aggregate
    mcg_agg = aggregate_proportions(mcg_props, "mCG DMGs")
    mch_agg = aggregate_proportions(mch_props, "mCH DMGs")
    all_agg = aggregate_proportions(all_props, "All Genes")

    # =========================================================================
    # Standard enrichment tests
    # =========================================================================
    print("\n" + "=" * 80)
    print("ENRICHMENT: mCG DMGs vs All Genes")
    print("=" * 80)
    mcg_enrich = enrichment_test_features(mcg_agg, all_agg, "mCG")
    print(mcg_enrich.to_string(index=False))

    print("\n" + "=" * 80)
    print("ENRICHMENT: mCH DMGs vs All Genes")
    print("=" * 80)
    mch_enrich = enrichment_test_features(mch_agg, all_agg, "mCH")
    print(mch_enrich.to_string(index=False))

    print("\n" + "=" * 80)
    print("DIRECT COMPARISON: mCG vs mCH DMGs")
    print("=" * 80)
    chi2_result, comparison_df = compare_mcg_vs_mch(mcg_agg, mch_agg)
    if comparison_df is not None:
        print(comparison_df.to_string(index=False))

    # =========================================================================
    # Length-matched analysis
    # =========================================================================
    print("\n" + "=" * 80)
    print("LENGTH-MATCHED ANALYSIS")
    print("=" * 80)

    print(f"\nGenerating {N_PERMUTATIONS} length-matched background samples for mCG DMGs...")
    mcg_matched_aggs = get_length_matched_background(mcg_props, all_props,
                                                      n_samples=N_PERMUTATIONS)
    print(f"  Generated {len(mcg_matched_aggs)} valid samples")

    print(f"\nGenerating {N_PERMUTATIONS} length-matched background samples for mCH DMGs...")
    mch_matched_aggs = get_length_matched_background(mch_props, all_props,
                                                      n_samples=N_PERMUTATIONS)
    print(f"  Generated {len(mch_matched_aggs)} valid samples")

    print("\nLength-matched enrichment: mCG DMGs")
    mcg_lm_enrich = length_matched_enrichment_test(mcg_agg, mcg_matched_aggs, "mCG")
    print(mcg_lm_enrich.to_string(index=False))

    print("\nLength-matched enrichment: mCH DMGs")
    mch_lm_enrich = length_matched_enrichment_test(mch_agg, mch_matched_aggs, "mCH")
    print(mch_lm_enrich.to_string(index=False))

    # Compute mean matched background for plotting
    mcg_matched_mean = {}
    mch_matched_mean = {}
    for fname in FEATURE_ORDER:
        mcg_matched_mean[fname] = np.mean([a[fname] for a in mcg_matched_aggs])
        mch_matched_mean[fname] = np.mean([a[fname] for a in mch_matched_aggs])

    # =========================================================================
    # Save results
    # =========================================================================
    mcg_enrich.to_csv(os.path.join(OUTPUT_DIR, "mcg_vs_allgenes_enrichment.csv"), index=False)
    mch_enrich.to_csv(os.path.join(OUTPUT_DIR, "mch_vs_allgenes_enrichment.csv"), index=False)
    if comparison_df is not None:
        comparison_df.to_csv(os.path.join(OUTPUT_DIR, "mcg_vs_mch_comparison.csv"), index=False)
    mcg_props.to_csv(os.path.join(OUTPUT_DIR, "mcg_dmgs_per_gene_proportions.csv"), index=False)
    mch_props.to_csv(os.path.join(OUTPUT_DIR, "mch_dmgs_per_gene_proportions.csv"), index=False)
    mcg_lm_enrich.to_csv(os.path.join(OUTPUT_DIR, "mcg_length_matched_enrichment.csv"), index=False)
    mch_lm_enrich.to_csv(os.path.join(OUTPUT_DIR, "mch_length_matched_enrichment.csv"), index=False)

    summary = pd.DataFrame({
        "Feature": FEATURE_ORDER,
        "mCG_pct": [mcg_agg[f]["pct_of_total_bp"] for f in FEATURE_ORDER],
        "mCH_pct": [mch_agg[f]["pct_of_total_bp"] for f in FEATURE_ORDER],
        "AllGenes_pct": [all_agg[f]["pct_of_total_bp"] for f in FEATURE_ORDER],
        "mCG_fold_vs_all": [mcg_enrich[mcg_enrich.Feature == f]["Fold_enrichment"].values[0] for f in FEATURE_ORDER],
        "mCH_fold_vs_all": [mch_enrich[mch_enrich.Feature == f]["Fold_enrichment"].values[0] for f in FEATURE_ORDER],
        "mCG_LengthMatched_mean": [mcg_matched_mean[f] for f in FEATURE_ORDER],
        "mCH_LengthMatched_mean": [mch_matched_mean[f] for f in FEATURE_ORDER],
    })
    summary.to_csv(os.path.join(OUTPUT_DIR, "summary_table.csv"), index=False)
    print(f"\nSaved all results to: {OUTPUT_DIR}/")

    # =========================================================================
    # Figures
    # =========================================================================
    print("\nGenerating figures...")
    plot_stacked_comparison(mcg_agg, mch_agg, all_agg,
                           os.path.join(OUTPUT_DIR, "stacked_bar_comparison.pdf"))
    plot_fold_enrichment_comparison(mcg_enrich, mch_enrich,
                                   os.path.join(OUTPUT_DIR, "fold_enrichment_grouped.pdf"))
    plot_per_gene_boxplots(mcg_props, mch_props, all_props,
                          os.path.join(OUTPUT_DIR, "per_gene_boxplots.pdf"))
    plot_gene_length_comparison(mcg_props, mch_props, all_props,
                               os.path.join(OUTPUT_DIR, "gene_length_comparison.pdf"))
    plot_length_matched_comparison(mcg_agg, mch_agg, mcg_matched_mean, mch_matched_mean,
                                  os.path.join(OUTPUT_DIR, "length_matched_comparison.pdf"))

    # =========================================================================
    # Summary
    # =========================================================================
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"  mCG DMGs: {len(mcg_df)} genes (median length: {mcg_props['gene_length'].median()/1000:.1f} kb)")
    print(f"  mCH DMGs: {len(mch_df)} genes (median length: {mch_props['gene_length'].median()/1000:.1f} kb)")
    print(f"  All genes: {len(all_genes_df)} genes (median length: {all_props['gene_length'].median()/1000:.1f} kb)")

    if chi2_result:
        chi2, chi2_p, dof = chi2_result
        sig_str = "SIGNIFICANT" if chi2_p < 0.05 else "not significant"
        print(f"\n  mCG vs mCH genomic context: chi2={chi2:.2f}, p={chi2_p:.2e} ({sig_str})")
        if chi2_p < 0.05:
            print("  -> mCG and mCH DMGs show DISTINCT genomic distributions,")
            print("     suggesting different regulatory mechanisms.")
        else:
            print("  -> mCG and mCH DMGs show SIMILAR genomic distributions,")
            print("     suggesting shared regulatory mechanisms.")

    # Length-matched summary
    print("\n  Length-matched analysis:")
    mcg_any_sig = mcg_lm_enrich["Significant"].any()
    mch_any_sig = mch_lm_enrich["Significant"].any()
    if mcg_any_sig:
        sig_feats = mcg_lm_enrich[mcg_lm_enrich["Significant"]]["Feature"].tolist()
        print(f"    mCG DMGs: Feature enrichment PERSISTS after length-matching for: {', '.join(sig_feats)}")
        print("    -> Enrichment is NOT solely driven by gene length.")
    else:
        print("    mCG DMGs: Feature enrichment is NOT significant after length-matching.")
        print("    -> Enrichment is likely driven by gene length differences.")

    if mch_any_sig:
        sig_feats = mch_lm_enrich[mch_lm_enrich["Significant"]]["Feature"].tolist()
        print(f"    mCH DMGs: Feature enrichment PERSISTS after length-matching for: {', '.join(sig_feats)}")
        print("    -> Enrichment is NOT solely driven by gene length.")
    else:
        print("    mCH DMGs: Feature enrichment is NOT significant after length-matching.")
        print("    -> Enrichment is likely driven by gene length differences.")

    print(f"\nDone! All results in: {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()
    

Loading DMG files...
  mCG DMGs: 75
  mCH DMGs: 643
Parsing GTF: ../globalmethyl/Macaca_mulatta.Mmul_10.104.gtf
  GTF chr prefix: False
  Genes: 35432, Exons: 605501, Transcripts: 64228
  5'UTRs: 51905, 3'UTRs: 32625
  Building priority-based features...
    Promoter: 38,965 regions, 106,044,858 bp
    5'UTR: 10,060 regions, 2,159,913 bp
    3'UTR: 19,229 regions, 22,744,894 bp
    Exon: 188,266 regions, 39,286,913 bp
    Intron: 208,313 regions, 1,221,066,363 bp

Harmonizing chromosome names...
  -> Stripping 'chr' prefix from DMG coordinates
  All genes in GTF: 35425

Computing feature proportions for mCG DMGs...

Computing feature proportions for mCH DMGs...

Computing feature proportions for all genes...
    Processing gene 5000/35425...
    Processing gene 10000/35425...
    Processing gene 15000/35425...
    Processing gene 20000/35425...
    Processing gene 25000/35425...
    Processing gene 30000/35425...
    Processing gene 35000/35425...

  mCG DMGs (n=75 genes, 286,276 total